# Reconnaître un schéma d'argumentation — la table de Walton et un classifieur déterministe

**Notebook pédagogique — série Argument_Analysis.** Ce carnet porte l'essence d'un organe du dépôt EPITA (mandat Triple Distillation) : la table des **schémas d'argumentation** de Walton et son **classifieur lexical déterministe**. La série dispose déjà du modèle *structurel* de Toulmin (`Toulmin_Model` : claim/data/warrant) et du cadre *abstrait* de Dung (`Dung_AF_Semantics` : arguments qui s'attaquent). Ici, le niveau intermédiaire : les **schémas stéréotypés** — autorité, analogie, cause à effet, consensus... — chacun avec ses prémisses types, sa conclusion type et surtout ses **questions critiques**, le test qu'un opposant applique pour prendre l'argument en défaut.

L'organe `argumentation_schemes.py` (port local, stdlib pur) ne comporte **aucun LLM, aucune JVM, aucun aléatoire** : un texte est classé par confrontation à des ensembles de mots-clés, et l'échec est **bruyant** — `None` signifie honnêtement « aucun schéma détecté », jamais une étiquette fabriquée.

**Provenance et contrat de fidélité.** La table et les forces a priori viennent verbatim du livrable étudiant `1_2_7_argumentation_dialogique` ; les questions critiques sont l'ajout canonique du tronc EPITA (le moteur étudiant n'en portait aucune). Le port documente les **divergences mesurées sur le source** au lieu de les corriger en silence : la docstring du source annonce un vocabulaire « FR + EN » alors que les 33 ensembles mesurés sont exclusivement français accentués (§4) ; la règle de la paire complète connait un singleton assumé (§6) ; et « donc » arme `cause_effect` avant `modus_ponens` (§5) — trois bornes du moteur d'origine, portées telles quelles et rendues visibles. La règle du dépôt s'applique : on restaure, on ne fabrique pas.

## §1 — Le modèle : dix schémas, une force a priori, des questions critiques

Chaque schéma est un **gabarit** : des prémisses types qui mènent à une conclusion type. La `strength` est la **force a priori** du moteur étudiant d'origine — un a priori sur la fiabilité du schéma, *pas* un score mesuré sur le texte. Les `critical_questions` sont les questions canoniques de Walton : ce qu'un challenger demande pour éprouver le schéma.

In [1]:
# L'organe est un module local pur (stdlib uniquement) — aucun LLM, aucune JVM.
import sys, os
sys.path.insert(0, os.path.abspath("."))

from argumentation_schemes import _load_argumentation_schemes

schemes = _load_argumentation_schemes()
print(f"{len(schemes)} schemas charges.\n")
print(f"{'cle':<24}{'force':>6}   {'label':<46}{'questions':>10}")
print("-" * 90)
for s in schemes.values():
    print(f"{s.key:<24}{s.strength:>6.2f}   {s.label:<46}{len(s.critical_questions):>10}")

10 schemas charges.

cle                      force   label                                          questions
------------------------------------------------------------------------------------------
modus_ponens              1.00   Déduction (modus ponens)                               2
expert_opinion            0.80   Argument d'autorité (advice of an expert)              3
analogy                   0.60   Argument par analogie                                  2
cause_effect              0.70   Argument de cause à effet                              2
consensus                 0.85   Argument d'ad hominem consensuel (appeal to consensus)         2
empirical_evidence        0.90   Argument empirique (from evidence)                     2
economic_argument         0.75   Argument économique (cost-benefit)                     2
precautionary_principle   0.70   Principe de précaution                                 2
moral_argument            0.80   Argument moral (from rights)         

**Lecture du résultat.** La table va de la déduction (`modus_ponens`, force `1.00` — le seul schéma conclusif : si les prémisses tiennent, la conclusion tient) à l'analogie (`analogy`, `0.60` — la forme la plus fragile du banc : la similitude n'entraine jamais la propriété par nécessité) : l'ordre des forces a priori épouse l'intuition argumentative classique, l'argument empirique (`0.90`) et le consensus d'experts (`0.85`) tenant le haut du pavé non déductif. Deux lectures d'attention : d'abord, la `strength` est un **a priori déclaré par le moteur étudiant**, pas une statistique mesurée sur un corpus — elle fixe l'ordre de confiance initial, pas un verdict ; ensuite, la colonne `questions` compte 2 questions critiques partout sauf l'autorité (`expert_opinion`, 3) : l'argument d'autorité est le seul du banc à exiger trois tests — expertise réelle, cohérence avec les pairs, preuve directe au-delà du témoignage. Le nombre de questions est lui-même une théorie de la fragilité : plus un schéma est testable, plus sa force dépend des réponses.

In [2]:
# Un schéma complet : prémisses types, conclusion type, questions critiques.
s = schemes["expert_opinion"]
print(f"Label : {s.label}")
print(f"Force a priori : {s.strength}")
print("\nPremisses types :")
for i, p in enumerate(s.premises_pattern, 1):
    print(f"  P{i}. {p}")
print(f"\nConclusion type : {s.conclusion_pattern}")
print("\nQuestions critiques (le test du challenger) :")
for i, q in enumerate(s.critical_questions, 1):
    print(f"  CQ{i}. {q}")

Label : Argument d'autorité (advice of an expert)
Force a priori : 0.8

Premisses types :
  P1. Expert E says P
  P2. E is expert in domain D
  P3. P is in domain D

Conclusion type : P

Questions critiques (le test du challenger) :
  CQ1. E est-elle réellement une source experte sur ce domaine ?
  CQ2. L'avis de E est-il cohérent avec le consensus des autres experts ?
  CQ3. Y a-t-il une preuve directe (au-delà du seul témoignage) ?


**Lecture du résultat.** Le gabarit de l'autorité se lit comme une checklist : trois prémisses (`E affirme P`, `E est experte du domaine D`, `P appartient à D`) dont la conjonction autorise la conclusion `P`. Classer un texte `expert_opinion` ne dit **pas** que l'argument est bon — cela dit de *quelle forme* il est, donc de quel test il doit passer : la question CQ3 (« y a-t-il une preuve directe au-delà du seul témoignage ? ») est exactement ce qui distingue un appel d'autorité légitime d'un sophisme d'autorité.

## §2 — Classifications réelles : le matcher départage dix textes

Le banc ci-dessous (verbatim du source EPITA) porte **un exemple par schéma**. Chaque texte est soumis au classifieur réel `classify_scheme` ; l'assertion de round-trip vérifie que le schéma attendu (`expected_key`) est bien celui qui est rendu — le carnet ne peut pas dériver silencieusement.

In [3]:
from argumentation_schemes import classify_scheme

BANC = [
    ("Selon un spécialiste reconnu, la molécule est sans danger pour l'usage courant.", "expert_opinion"),
    ("L'étude repose sur une mesure directe et un échantillon représentatif du public.", "empirical_evidence"),
    ("La majorité des chercheurs parviennent à un accord sur ce point.", "consensus"),
    ("À l'instar du pilote maritime, comparable à un navigateur solitaire, le capitaine doit tracer sa route.", "analogy"),
    ("Cette cause profonde produit un effet mesurable sur le rendement.", "cause_effect"),
    ("Le coût total reste faible au regard du bénéfice annuel obtenu.", "economic_argument"),
    ("Compte tenu du risque identifié, la prévention s'impose dès maintenant.", "precautionary_principle"),
    ("Cette atteinte aux droits fondamentaux constitue une violation caractérisée.", "moral_argument"),
    ("Il existe un précédent historique : la même configuration a déjà été observée.", "historical_precedent"),
    ("La règle l'implique ; par conséquent, si la prémisse est établie, la conclusion suit.", "modus_ponens"),
]

ok = 0
for text, expected in BANC:
    scheme = classify_scheme(text)
    assert scheme is not None and scheme.key == expected, (text, expected, scheme)
    ok += 1
    print(f"{expected:<24} <- {text[:62]}")
print(f"\n{ok}/10 exemples classes selon l'attendu (round-trip verifie).")

expert_opinion           <- Selon un spécialiste reconnu, la molécule est sans danger pour
empirical_evidence       <- L'étude repose sur une mesure directe et un échantillon représ
consensus                <- La majorité des chercheurs parviennent à un accord sur ce poin
analogy                  <- À l'instar du pilote maritime, comparable à un navigateur soli
cause_effect             <- Cette cause profonde produit un effet mesurable sur le rendeme
economic_argument        <- Le coût total reste faible au regard du bénéfice annuel obtenu
precautionary_principle  <- Compte tenu du risque identifié, la prévention s'impose dès ma
moral_argument           <- Cette atteinte aux droits fondamentaux constitue une violation
historical_precedent     <- Il existe un précédent historique : la même configuration a dé
modus_ponens             <- La règle l'implique ; par conséquent, si la prémisse est établ

10/10 exemples classes selon l'attendu (round-trip verifie).


**Lecture du résultat.** Dix textes, dix schémas, aucune erreur : chaque texte du banc porte la **paire complète** de mots-clés qui signe son schéma — « selon + spécialiste » pour l'autorité, « étude + mesure » pour l'empirique, « à l'instar + comparable » pour l'analogie. La paire joue le rôle d'une signature : chacun des mots seul est banal (« mesure » apparaît dans une recette de cuisine, « spécialiste » dans une annonce de conférence), leur co-occurrence dans un même texte est le signal argumentatif. Remarquez la prudence du banc pour `modus_ponens` : son exemple utilise « par conséquent + si » et évite soigneusement « donc », dont on verra au §5 qu'il arme aussi `cause_effect` — le banc contourne l'écueil au lieu de le cacher, et l'exercice 3 vous demande de le ré-armer vous-même. Dernier point de mesure : ce banc n'exerce qu'**un ensemble de mots-clés par schéma** (10 des 33 du moteur, ~30 %) — toujours le même profil de paire canonique ; les paires alternatives sont laissées à l'exercice 1, et les tests automatisés de l'organe en couvrent d'autres.

## §3 — L'honnêteté du `None` : quatre textes sans schéma

Le matcher exige la **paire complète** : un mot-clé isolé ne tire jamais. Quand rien ne tire, le classifieur rend `None` — un « aucun schéma détecté » **honnête**, jamais une étiquette fabriquée. C'est le contrat d'échec bruyant de l'organe.

In [4]:
NEGATIFS = [
    "Bonjour, il fait beau aujourd'hui.",
    "Le rapport compte douze pages et trois annexes.",
    "L'expert est attendu à la conférence.",
    "",
]

for text in NEGATIFS:
    scheme = classify_scheme(text)
    assert scheme is None, (text, scheme)
    print(f"None <- {text!r}")

print("\n4/4 negatifs rendent None (honetete du classifieur).")

None <- "Bonjour, il fait beau aujourd'hui."
None <- 'Le rapport compte douze pages et trois annexes.'
None <- "L'expert est attendu à la conférence."
None <- ''

4/4 negatifs rendent None (honetete du classifieur).


**Lecture du résultat.** Le troisième négatif est le plus instructif : « L'**expert** est attendu à la conférence » contient bien un mot-clé du schéma d'autorité — mais isolé, sans son binôme (« domaine », « spécialiste », « source »...). Le classifieur préfère ne rien dire plutôt que de mal étiqueter : c'est le choix **précision d'abord**, assumé dans la conception. La conséquence opérationnelle est à écrire noir sur blanc : sur un corpus de textes neutres (météo, comptes rendus, petites annonces), ce classifieur rendra majoritairement `None` — et c'est le comportement **voulu**. Un outil de classification d'argument qui étiquetterait « il fait beau » comme analogie serait inutilisable ; l'honnêteté du `None` est ce qui rend les étiquettes positives crédibles quand elles arrivent. Le texte vide, lui, rend `None` immédiatement, sans même parcourir la table — la garde minimale avant tout parcours.

## §4 — La limite assumée : les mots-clés sont accentués

Les 33 ensembles de mots-clés de la table sont exclusivement **français accentués**. La docstring du source original annonce un vocabulaire « FR + EN » — mesure faite, aucun mot anglais ne figure dans la table, et un texte désaccentué casse la paire. Le port documente la limite au lieu de la corriger : c'est une borne du moteur d'origine, pas un défaut du port.

In [5]:
# Texte desaccentue : la paire « selon + spécialiste » ne se reforme pas.
texte_desaccentue = "Selon ce specialiste reconnu, la molecule est sans danger."
print(f"classify_scheme(texte_desaccentue) = {classify_scheme(texte_desaccentue)}")
assert classify_scheme(texte_desaccentue) is None

# Mot isole : jamais de tir, quelle que soit la paire dont il est extrait.
for mot in ["donc", "si", "comme", "expert", "droit", "mesure"]:
    assert classify_scheme(mot) is None, mot
print("6 mots isoles testes : aucun ne tire seul.")

classify_scheme(texte_desaccentue) = None
6 mots isoles testes : aucun ne tire seul.


**Lecture du résultat.** « specialiste » sans accent ne contient pas la sous-chaîne « spécialiste » : la paire ne se reforme pas et le texte rend `None`. C'est la contrepartie du choix de sous-chaînes accentuées — le même mécanisme qui protège la précision (le §3) rend le classifieur sensible à la graphie exacte. Un pipeline amont qui normalise les accents doit en être conscient : il perd alors tout le banc.

## §5 — Le départage : l'ordre canonique et le piège de « donc »

Un texte peut armer **plusieurs** schémas à la fois. Le classifieur les départage par un **ordre canonique** : les schémas spécifiques d'abord, `modus_ponens` en dernier — parce que « donc » est partout, ce signal ne doit tirer que quand rien de plus spécifique n'a tiré. La fonction `match_report` expose ce mécanisme : elle rend *tous* les schémas qui tirent, dans l'ordre du départage.

In [6]:
from argumentation_schemes import match_report

# Texte ambigu : arme expert_opinion ET modus_ponens.
ambigu = "Selon ce spécialiste, la règle implique donc la conclusion."
report = match_report(ambigu)
for r in report:
    print(f"rang {r['rank']:>2} : {r['key']:<18} arme par {r['matched_keywords']}")
print(f"\nclassify_scheme retient : {classify_scheme(ambigu).key}")
assert classify_scheme(ambigu).key == "expert_opinion"
assert [r["key"] for r in report] == ["expert_opinion", "modus_ponens"]

# « donc » arme AUSSI cause_effect (paire « donc + parce que »), placé AVANT modus_ponens.
piege = "Il pleut parce que le nuage est là, donc le sol est humide."
print(f"\nPiege cause/modus : classify_scheme retient {classify_scheme(piege).key}")
assert classify_scheme(piege).key == "cause_effect"  # la paire arme cause_effect avant modus_ponens


rang  1 : expert_opinion     arme par ['selon', 'spécialiste']
rang 10 : modus_ponens       arme par ['donc', 'implique']

classify_scheme retient : expert_opinion

Piege cause/modus : classify_scheme retient cause_effect


**Lecture du résultat.** Le rapport montre deux candidats sur le texte ambigu : l'autorité (rang 1) et le modus ponens (rang 10) — et le classifieur retient l'autorité, la plus spécifique. `match_report` n'est pas un second classifieur : c'est la **même mécanique exposée** (le premier élément du rapport est toujours ce que `classify_scheme` rend), un instrument d'inspection qui rend visible un choix sinon invisible. Le second cas révèle la mesure fine de l'ordre : la paire « donc + parce que » arme `cause_effect`, placé *avant* `modus_ponens` — un texte causal-déductif est donc classé causal, et le modus ponens lexical ne tire que sur ses propres paires (« donc + implique », « par conséquent + si »). L'ordre canonique n'est pas une préférence esthétique : c'est lui qui décide de l'étiquette finale dès qu'un texte arme plusieurs schémas — et une inversion de cet ordre changerait des étiquettes sans toucher un seul mot-clé.

## §6 — L'unique singleton : « principe de précaution »

La règle de la §3 (paire complète) connait **un seul écart**, mesuré sur le source : l'ensemble `["principe de précaution"]` — la locution entière suffit à armer le schéma, sans binôme. Écart porté tel quel : il est volontaire (une locution figée est aussi discriminante qu'une paire) et documenté ici au lieu d'être corrigé.

In [7]:
texte = "Nous invoquons le principe de précaution."
scheme = classify_scheme(texte)
print(f"classify_scheme retient : {scheme.key}")
assert scheme.key == "precautionary_principle"
print("La locution entiere tire seule : l'unique singleton de la table (33 ensembles, 32 paires, 1 singleton).")

classify_scheme retient : precautionary_principle
La locution entiere tire seule : l'unique singleton de la table (33 ensembles, 32 paires, 1 singleton).


**Lecture du résultat.** Le singleton passe parce que la locution « principe de précaution » est **figée** : sa présence dans un texte est déjà un signal argumentatif complet, là où « expert » seul ne dit rien (le mot apparaît dans une annonce de conférence, cf. §3). La table encode ainsi une gradation implicite : plus le mot-clé est banal, plus il exige de compagnie.

## §7 — Les questions critiques : de la classification au test

Classer **nomme** le schéma ; les questions critiques le **testent**. C'est la brique qui manquait au moteur étudiant d'origine — la table portait les forces mais aucune question : le tronc EPITA y a ajouté les questions canoniques de Walton. L'audit ci-dessous montre l'enchaînement complet sur un texte réel : forme reconnue, tests applicables, force a priori.

In [8]:
texte = "L'étude repose sur une mesure directe et un échantillon représentatif du public."
scheme = classify_scheme(texte)
print(f"Texte classé : {scheme.label} (force a priori {scheme.strength:.2f})")
print("\nAudit — les tests que ce texte doit passer :")
for i, q in enumerate(scheme.critical_questions, 1):
    print(f"  CQ{i}. {q}")

Texte classé : Argument empirique (from evidence) (force a priori 0.90)

Audit — les tests que ce texte doit passer :
  CQ1. Les données sont-elles fiables (collecte, mesure) ?
  CQ2. L'échantillon est-il représentatif de la population visée ?


**Lecture du résultat.** L'argument empirique est reconnu à sa paire « étude + mesure », et l'audit liste ses deux questions canoniques : fiabilité des données, représentativité de l'échantillon. Un texte peut porter la *forme* parfaite d'un argument empirique et échouer à ses deux tests (données falsifiées, échantillon biaisé) — la classification ouvre l'évaluation, elle ne la remplace pas. C'est la même discipline que la matrice de richesse formelle (`Formal_Richness_Matrix`) appliquée cette fois à l'argumentation informelle : nommer la forme, puis demander compte. Et le contraste avec le détecteur de sophismes de la série prend ici son sens : détecter un sophisme dit ce qui *cloche* ; classifier un schéma dit quelle *force* l'argument prétend avoir — les deux questions ne se confondent pas, un argument d'autorité parfaitement formel reste un argument d'autorité, à tester comme tel. Le lien avec les dialogues protocolisés est le même : les protocoles d'inquiry/persuasion organisent *qui parle quand*, les schémas organisent *ce que vaut chaque coup*.

## §8 — La table comme contexte : le rendu textuel

Côté EPITA, le débat multi-agents consomme la table via `schemes_as_prompt_context` : un rendu textuel borné que le prompt LLM reçoit pour ancrer ses échanges sur des schémas réels. Côté CoursIA, ce rendu sert de lecture condensée de la table — et en révèle une mesure fine : la troncature à deux questions par schéma.

In [9]:
from argumentation_schemes import schemes_as_prompt_context

ctx = schemes_as_prompt_context()
print(f"{len(ctx.splitlines())} lignes rendues :")
print(ctx)
assert len(ctx.splitlines()) == 10

10 lignes rendues :
  1. « Déduction (modus ponens) » (force a priori 1.00) — questions critiques de test : La prémisse P est-elle effectivement établie ? ; L'implication P → Q est-elle valide (pas un sophisme conditionnel) ?
  2. « Argument d'autorité (advice of an expert) » (force a priori 0.80) — questions critiques de test : E est-elle réellement une source experte sur ce domaine ? ; L'avis de E est-il cohérent avec le consensus des autres experts ?
  3. « Argument par analogie » (force a priori 0.60) — questions critiques de test : En quoi les cas A et B sont-ils réellement similaires sur la dimension pertinente ? ; Existe-t-il une différence pertinente qui brise l'analogie ?
  4. « Argument de cause à effet » (force a priori 0.70) — questions critiques de test : La relation causale A → B est-elle établie (et non une simple corrélation) ? ; Y a-t-il d'autres causes possibles de B ?
  5. « Argument d'ad hominem consensuel (appeal to consensus) » (force a priori 0.85) — questions cr

**Lecture du résultat.** Le rendu porte les dix schémas avec force a priori et deux questions chacun — mais la **troisième** question critique de l'autorité (« y a-t-il une preuve directe au-delà du seul témoignage ? ») n'apparaît pas : le rendu du source tronque à `[:2]`, porté tel quel. Mesure visible dans le carnet lui-même : la table du §1 compte `questions: 3` pour l'autorité, ce rendu n'en montre que deux. Dans le pipeline LLM d'origine, cette économie de budget de prompt se paye d'un test perdu — leçons des deux côtés du pont.

## Exercice 1 — Armer une paire alternative

La table d'autorité compte **quatre** ensembles de mots-clés ; le banc du §2 n'en exerce qu'un (« selon + spécialiste »). Écrivez dans `mon_texte` un texte court qui arme la paire `["source", "expert"]` — le classifieur doit rendre `expert_opinion`. *Indice : les deux mots doivent apparaître, n'importe où dans le texte.*

In [10]:
# Exercice a completer : la paire « source + expert » doit classer le texte en autorite.
mon_texte = ""  # TODO etudiant : votre texte ici.

resultat = classify_scheme(mon_texte)
print(f"classify_scheme(mon_texte) = {resultat.key if resultat else None}")

classify_scheme(mon_texte) = None


## Exercice 2 — Un négatif résistant

Le §3 a montré qu'un mot isolé ne tire jamais. Compliquons : écrivez dans `mon_defi` un texte qui contient **un mot-clé de deux schémas différents** (par exemple « expert » et « mesure ») et qui pourtant rend `None`. *Indice : chaque paire exige ses DEUX mots — un mot de chaque paire ne suffit pas.*

In [11]:
# Exercice a completer : un mot de deux schemas differents, et pourtant None.
mon_defi = ""  # TODO etudiant : votre texte ici.

resultat_defi = classify_scheme(mon_defi)
print(f"classify_scheme(mon_defi) = {resultat_defi.key if resultat_defi else None} (attendu : None)")

classify_scheme(mon_defi) = None (attendu : None)


## Exercice 3 — Prédire le rapport avant de le lancer

Avant d'exécuter la cellule, écrivez votre prédiction : quels schémas `match_report` va-t-il lister sur le texte `mon_ambigu` ci-dessous, et dans quel ordre ? Complétez `prediction` (une liste de clés), puis exécutez : la cellule affiche votre prédiction à côté du rapport réel. *Indice : repérez toutes les paires complètes présentes dans le texte, puis appliquez l'ordre canonique du §5.*

In [12]:
# Exercice a completer : predire les candidats AVANT l'execution.
mon_ambigu = "Selon la source experte, le risque justifie la prévention, donc l'étude impose la mesure."
prediction = []  # TODO etudiant : liste attendue des cles, dans l'ordre du rapport.

rapport = match_report(mon_ambigu)
print(f"Rapport reel   : {[r['key'] for r in rapport]}")
print(f"Votre prediction : {prediction}")

Rapport reel   : ['expert_opinion', 'empirical_evidence', 'precautionary_principle']
Votre prediction : []


## Ce qu'il faut retenir

- **Dix schémas de Walton** : prémisses types, conclusion type, force a priori (1.00 pour la déduction, 0.60 pour l'analogie) et questions critiques canoniques — la table est celle du moteur étudiant EPITA, les questions sont l'ajout du tronc.
- **Un classifieur lexical déterministe** : 33 ensembles de mots-clés français accentués, paire complète exigée (un singleton : « principe de précaution »), ordre canonique qui départage — `modus_ponens` en dernier car « donc » arme aussi `cause_effect`.
- **L'échec est bruyant** : `None` est un résultat honnête, jamais une étiquette fabriquée. Un mot isolé ne tire pas ; un texte désaccentué ne tire pas.
- **Classer nomme, les questions testent** : l'audit d'un texte classé liste les tests canoniques qu'il doit passer — la classification ouvre l'évaluation, elle ne la remplace pas.
- **Position dans la série** : `Toulmin_Model` déplie la *structure* d'un argument, `Dung_AF_Semantics` calcule *qui survit* aux attaques ; ce carnet identifie *la forme stéréotypée* d'un texte et fournit son test — les trois niveaux de l'argumentation informelle à la computationnelle.
- **Les limites mesurées, portées au grand jour** : vocabulaire exclusivement français accentué (un texte désaccentué rend `None`, §4), un singleton assumé (§6), « donc » partagé entre deux schémas (§5), rendu tronqué à deux questions (§8). Aucune de ces bornes n'est un défaut du port : chacune est une mesure du moteur d'origine, documentée pour que l'utilisateur du classifieur sache exactement ce qu'il tient — et ce qu'il ne tient pas.